## Undersök Datasetet

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

#Ladda in datasetet
df = pd.read_csv('Dataset/train.csv')
print("\n=== Sammanfattning av Data ===")
print(df.info())
print("\n=== Datatyper och unika värden per kolumn ===")
for col in df.columns:
    print(f"{col}: {df[col].dtype}, {df[col].nunique()} unika värden")
print("--------------------------------")
print("Dubblikater i Data:", df.duplicated().sum())
print("Missing value detection:",df.isnull().sum().sum())





=== Sammanfattning av Data ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 913000 entries, 0 to 912999
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   date    913000 non-null  object
 1   store   913000 non-null  int64 
 2   item    913000 non-null  int64 
 3   sales   913000 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 27.9+ MB
None

=== Datatyper och unika värden per kolumn ===
date: object, 1826 unika värden
store: int64, 10 unika värden
item: int64, 50 unika värden
sales: int64, 213 unika värden
--------------------------------
Dubblikater i Data: 0
Missing value detection: 0


## Hantering av dubbletter, saknade värden och typkonvertering

In [3]:
# Kör minimala datakontroller
import pandas as pd

def run_min_checks(df: pd.DataFrame, date_format: str = "%Y-%m-%d"):
    issues = {}
    # Kolumner
    required = ["date", "store", "item", "sales"]
    missing_cols = [c for c in required if c not in df.columns]
    issues["missing_columns"] = missing_cols

    # Typkonvertering (tålig) + kopia
    d = df.copy()
    d["date"] = pd.to_datetime(d["date"], format=date_format, errors="coerce")
    d["store"] = pd.to_numeric(d["store"], errors="coerce")
    d["item"]  = pd.to_numeric(d["item"],  errors="coerce")
    d["sales"] = pd.to_numeric(d["sales"], errors="coerce")

    type_fail = {
        "date": int(d["date"].isna().sum()),
        "store": int(d["store"].isna().sum()),
        "item":  int(d["item" ].isna().sum()),
        "sales": int(d["sales"].isna().sum()),
    }
    issues["type_conversion_nulls"] = type_fail

    # Nulls (efter konvertering)
    nulls = d.isna().sum().to_dict()
    issues["nulls_per_column"] = {k:int(v) for k,v in nulls.items()}

    # Nyckelunikhet
    dup_cnt = int(d.duplicated(["date","store","item"]).sum())
    issues["key_duplicates"] = dup_cnt

    # Härled förväntad cardinalitet
    n_stores = int(d["store"].nunique())
    n_items  = int(d["item" ].nunique())
    n_dates  = int(d["date" ].nunique())
    issues["cardinality"] = {"stores": n_stores, "items": n_items, "dates": n_dates}

    # Per-dag fullständighet
    expected_per_day = n_stores * n_items
    per_day = d.groupby("date").size()
    bad_days = per_day[per_day != expected_per_day]
    issues["bad_days_count"] = int(bad_days.size)
    issues["bad_days_examples"] = [str(idx.date()) for idx in bad_days.index[:5]]

    # Volym sanity
    total_expected = expected_per_day * n_dates
    issues["total_rows"] = int(len(d))
    issues["total_expected"] = int(total_expected)
    issues["total_match"] = (len(d) == total_expected)

    # Sales rimlighet
    nonneg_fail = int((d["sales"] < 0).sum())
    upper = float(d["sales"].quantile(0.999) * 1.5)
    outliers_high = int((d["sales"] > upper).sum())
    issues["sales_checks"] = {
        "nonneg_fail": nonneg_fail,
        "upper_bound": upper,
        "outliers_high": outliers_high,
    }

    # Print sammanfattning
    print("=== Minimal Data Quality Checks ===")
    print("Missing columns:", issues["missing_columns"])
    print("Type conversion nulls:", issues["type_conversion_nulls"])
    print("Nulls per column:", issues["nulls_per_column"])
    print("Key duplicates:", issues["key_duplicates"])
    print("Cardinality:", issues["cardinality"])
    print(f"Per-day expected {expected_per_day}, bad days:", issues["bad_days_count"])
    print("Bad day examples:", issues["bad_days_examples"])
    print("Total rows:", issues["total_rows"], "| Total expected:", issues["total_expected"], "| Match:", issues["total_match"])
    print("Sales checks:", issues["sales_checks"])
    return issues

checks = run_min_checks(df, date_format="%Y-%m-%d")

=== Minimal Data Quality Checks ===
Missing columns: []
Type conversion nulls: {'date': 0, 'store': 0, 'item': 0, 'sales': 0}
Nulls per column: {'date': 0, 'store': 0, 'item': 0, 'sales': 0}
Key duplicates: 0
Cardinality: {'stores': 10, 'items': 50, 'dates': 1826}
Per-day expected 500, bad days: 0
Bad day examples: []
Total rows: 913000 | Total expected: 913000 | Match: True
Sales checks: {'nonneg_fail': 0, 'upper_bound': 246.0, 'outliers_high': 0}


## Skapa nya features

Koden skapar nya features (variabler) som kan användas i en maskininlärningsmodell för att förutsäga **sales**:

1. **Datum-baserade features**: veckodag, månad, kvartal, helg, månadsstart/-slut
2. **Cykliska features**: sin/cos-transformationer för veckodag och månad
3. **Lag-features**: försäljning 1, 7 och 365 dagar bakåt per (store, item)
4. **Rullande medelvärde**: 7-dagars snitt (utan dataläckage)
5. **Momentum**: vecka-över-vecka förändring
6. **Butiksnivå**: genomsnittlig försäljning för butiken föregående dag
7. **Svenska helgdagar**: flagga för röda dagar

In [4]:
import numpy as np
from workalendar.europe import Sweden

# Använd befintlig df från tidigare cell (undvik dubbel inläsning)
# Datum-baserade features
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
df = df.sort_values(["store", "item", "date"]).reset_index(drop=True)

df["dow"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["quarter"] = df["date"].dt.quarter
df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)

df["is_weekend"] = (df["dow"] >= 5).astype(int)
df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)

# Gör tid cyklisk istället för linjär (viktigt för ML-modeller)
# t.ex. att söndag och måndag samt december och januari ligger nära varandra i tid
df["dow_sin"] = np.sin(2 * np.pi * df["dow"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dow"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# Lag-features: historisk försäljning per (store, item)
df["sales_lag_1"] = df.groupby(["store", "item"])["sales"].shift(1)
df["sales_lag_7"] = df.groupby(["store", "item"])["sales"].shift(7)
df["sales_lag_365"] = df.groupby(["store", "item"])["sales"].shift(365)

# Week-over-week momentum
df["wow_change"] = (df["sales_lag_1"] - df["sales_lag_7"]) / df["sales_lag_7"].replace(0, np.nan)

# Store-level snitt från FÖREGÅENDE dag (undviker dataläckage)
store_daily = df.groupby(["store", "date"])["sales"].mean()
df["store_daily_avg_lag1"] = df.apply(
    lambda r: store_daily.get((r["store"], r["date"] - pd.Timedelta(days=1)), np.nan),
    axis=1
)

# Swedish holidays
cal = Sweden()
df["is_holiday"] = df["date"].apply(lambda d: cal.is_holiday(d)).astype(int)

# Rullande medel 7 dagar bakåt per (store, item) utan läckage
'''
Här byggs ett 7-dagars rullande medelvärde per (store, item):
    - Viktigt: man gör först shift(1) → då används bara historiska dagar, inte dagens sales.
    - Det undviker dataläckage (att feature råkar använda information från framtiden/nutid som modellen inte skulle ha vid prediktion).
    - rolling(window=7) tar snitt över de senaste 7 observationerna (bakåt i tiden).
    - min_periods=1 gör att den kan beräkna snitt även tidigt i serien (med färre än 7 dagar).
'''
sales_shifted = df.groupby(["store", "item"], observed=True)["sales"].shift(1)
roll_mean_7 = (
    sales_shifted
    .groupby([df["store"], df["item"]], observed=True)
    .rolling(window=7, min_periods=1)
    .mean()
    .reset_index(level=[0, 1], drop=True)
)
df["roll_mean_7"] = roll_mean_7

created_cols = [
    "dow", "month", "day", "quarter", "weekofyear",
    "is_weekend", "is_month_start", "is_month_end",
    "dow_sin", "dow_cos", "month_sin", "month_cos",
    "sales_lag_1", "sales_lag_7", "sales_lag_365",
    "wow_change", "store_daily_avg_lag1", "is_holiday", "roll_mean_7"
]
print("Feature columns created:", created_cols)

df.head()

Feature columns created: ['dow', 'month', 'day', 'quarter', 'weekofyear', 'is_weekend', 'is_month_start', 'is_month_end', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'sales_lag_1', 'sales_lag_7', 'sales_lag_365', 'wow_change', 'store_daily_avg_lag1', 'is_holiday', 'roll_mean_7']


,date,store,item,sales,dow,month,day,quarter,weekofyear,is_weekend,is_month_start,is_month_end,dow_sin,dow_cos,month_sin,month_cos,sales_lag_1,sales_lag_7,sales_lag_365,wow_change,store_daily_avg_lag1,is_holiday,roll_mean_7
0,2013-01-01,1,1,13,1,1,1,1,1,0,1,0,0.781831,0.623490,0.5,0.866025,NaN,NaN,NaN,NaN,NaN,1,NaN
1,2013-01-02,1,1,11,2,1,2,1,1,0,0,0,0.974928,-0.222521,0.5,0.866025,13.0,NaN,NaN,NaN,26.32,0,13.000000
2,2013-01-03,1,1,14,3,1,3,1,1,0,0,0,0.433884,-0.900969,0.5,0.866025,11.0,NaN,NaN,NaN,25.28,0,12.000000
3,2013-01-04,1,1,13,4,1,4,1,1,0,0,0,-0.433884,-0.900969,0.5,0.866025,14.0,NaN,NaN,NaN,26.10,0,12.666667
4,2013-01-05,1,1,10,5,1,5,1,1,1,0,0,-0.974928,-0.222521,0.5,0.866025,13.0,NaN,NaN,NaN,29.04,0,12.750000


## Train/Validation Split

Tidsserie-data kräver temporal split för att undvika dataläckage. Vi använder de sista 90 dagarna som valideringsdata.

In [6]:
def temporal_train_val_split(df: pd.DataFrame, val_days: int = 90):
    """
    Tidsserie-baserad split som undviker dataläckage.
    Valideringsdata är alltid EFTER träningsdata i tid.
    """
    cutoff = df["date"].max() - pd.Timedelta(days=val_days)
    train = df[df["date"] <= cutoff].copy()
    val = df[df["date"] > cutoff].copy()
    return train, val

train_df, val_df = temporal_train_val_split(df, val_days=90)

print(f"Train period: {train_df['date'].min().date()} to {train_df['date'].max().date()}")
print(f"Val period:   {val_df['date'].min().date()} to {val_df['date'].max().date()}")
print(f"Train rows:   {len(train_df):,}")
print(f"Val rows:     {len(val_df):,}")

Train period: 2013-01-01 to 2017-10-02
Val period:   2017-10-03 to 2017-12-31
Train rows:   868,000
Val rows:     45,000


## Baseline Model

En enkel baseline för att jämföra framtida modeller mot. Vi använder "gårdagens försäljning" som prediktion (naiv baseline).

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Filtrera bort rader med NaN i lag-features för rättvis jämförelse
val_clean = val_df.dropna(subset=["sales_lag_1", "roll_mean_7"])

# Baseline 1: Gårdagens försäljning
baseline_lag1 = val_clean["sales_lag_1"]
mae_lag1 = mean_absolute_error(val_clean["sales"], baseline_lag1)
rmse_lag1 = root_mean_squared_error(val_clean["sales"], baseline_lag1)

# Baseline 2: 7-dagars rullande medelvärde
baseline_roll7 = val_clean["roll_mean_7"]
mae_roll7 = mean_absolute_error(val_clean["sales"], baseline_roll7)
rmse_roll7 = root_mean_squared_error(val_clean["sales"], baseline_roll7)

# Baseline 3: Global genomsnitt (svagaste baseline)
global_mean = train_df["sales"].mean()
mae_mean = mean_absolute_error(val_clean["sales"], [global_mean] * len(val_clean))
rmse_mean = root_mean_squared_error(val_clean["sales"], [global_mean] * len(val_clean))

print("=== Baseline Results (Validation Set) ===")
print(f"{'Model':<25} {'MAE':>10} {'RMSE':>10}")
print("-" * 47)
print(f"{'Global Mean':<25} {mae_mean:>10.2f} {rmse_mean:>10.2f}")
print(f"{'Lag-1 (yesterday)':<25} {mae_lag1:>10.2f} {rmse_lag1:>10.2f}")
print(f"{'Rolling Mean 7-day':<25} {mae_roll7:>10.2f} {rmse_roll7:>10.2f}")
print()
print(f"Validation samples: {len(val_clean):,}")